In [1]:
import pandas as pd
import tqdm, os
import pickle
import warnings
warnings.simplefilter("ignore")
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, Subset

import sys
sys.path.insert(0, "../../..")

from utils.scaling import fit_scalers, apply_scaler
from utils.data_processing import dataset_df_to_token_df
from utils.acoustics_processing import get_mfccs
from utils.articulation_processing import get_frmri_articulatory_features

In [2]:
df = pd.read_csv("../../data_processing/dataset_construction/datasets/processed_frmri.csv")
df["subject"] = df["art_f"].apply(lambda x: os.path.basename(x).split("_")[1])
token_df = dataset_df_to_token_df(df)

In [3]:
# token_df = token_df.sample(frac=1).reset_index(drop=True)
# token_df = token_df.iloc[int(0.95 * len(token_df)):]
# token_df.sort_values(by='start', inplace=True)
# token_df.sort_values(by='ac_f', inplace=True)

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
import numpy as np
from utils.models import MDN
import copy

from utils.models import SequenceAutoencoder, VideoAutoencoder

device = "cpu"

art_model = VideoAutoencoder(
    input_hw=(136, 136),
    latent_dim=64,
    input_channels=1,
    output_activation=None,
    stn_mode="shared",
    stn_transform="rigid",
).to(device)

ckpt = torch.load(
    "../../encoder_training/ckpts/frmri/art_model_ckpt.pth",
    map_location=device,
)
art_model.load_state_dict(ckpt["model"])
art_model.eval()


ac_model = SequenceAutoencoder(
    feature_shape=13,
    latent_dim=64,
).to(device)
ac_model.load_state_dict(torch.load("../../encoder_training/ckpts/frmri/ac_model_ckpt.pth", map_location=device))
ac_model.eval()

SequenceAutoencoder(
  (encoder_cnn): TimeMargin(
    (body): Sequential(
      (0): Conv1d(13, 128, kernel_size=(3,), stride=(1,), padding=(1,))
      (1): FrameNorm(
        (gn): GroupNorm(8, 128, eps=1e-05, affine=True)
      )
      (2): ReLU()
      (3): Conv1d(128, 128, kernel_size=(3,), stride=(1,), padding=(1,))
      (4): FrameNorm(
        (gn): GroupNorm(8, 128, eps=1e-05, affine=True)
      )
      (5): ReLU()
      (6): Conv1d(128, 128, kernel_size=(3,), stride=(1,), padding=(1,))
      (7): FrameNorm(
        (gn): GroupNorm(8, 128, eps=1e-05, affine=True)
      )
      (8): ReLU()
    )
  )
  (encoder_proj): Linear(in_features=128, out_features=256, bias=True)
  (trunk): TemporalTrunk(
    (encoder_gru): GRU(256, 256, batch_first=True, bidirectional=True)
    (pool): MaskedPool(
      (score): Sequential(
        (0): Linear(in_features=512, out_features=256, bias=True)
        (1): Tanh()
        (2): Linear(in_features=256, out_features=1, bias=True)
      )
    )
   

In [5]:
import cv2
with open("../../encoder_training/scalers/frmri_video_stats.pkl", "rb") as f:
    _REC_STATS = pickle.load(f)

def recording_stats(path, n_sample=400):
    """
    Pixel mean and sd over the WHOLE recording, cached per file.
 
    Note cv2's CAP_PROP_POS_FRAMES seeking on compressed video snaps to the
    nearest keyframe on some codecs. That is harmless for statistics, but it is
    worth checking separately that _read_video_segment returns frames that really
    correspond to the requested times — a silent half-second offset would
    misalign every token against the audio.
    """
    if path in _REC_STATS:
        return _REC_STATS[path]
    cap = cv2.VideoCapture(path)
    if not cap.isOpened():
        cap.release()
        return None
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total <= 0:
        cap.release()
        return None
    idxs = np.unique(np.linspace(0, total - 1, min(n_sample, total)).astype(int))
    vals = []
    for i in idxs:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(i))
        ok, fr = cap.read()
        if ok:
            vals.append(fr[:, :, 0].astype(np.float32))
    cap.release()
    if not vals:
        return None
    v = np.stack(vals)
    _REC_STATS[path] = (float(v.mean()), float(v.std()) + 1e-5)
    return _REC_STATS[path]

with open(f"../../encoder_training/scalers/frmri_ac.pkl", "rb") as f:
    ac_scalers = pickle.load(f)

In [6]:
art_vectors = []
ac_vectors = []
phonemes = []
subjects = []
groups = []

pbar = tqdm.tqdm(token_df.iterrows(), total=len(token_df))

batch = []
batch_idx = 0
for idx, (_, data) in enumerate(pbar):

    start, end, phone, art_f, ac_f, subject = data["start"], data["end"], data["phone"], data["art_f"], data["ac_f"], data["subject"]
    stats = recording_stats(art_f)
    articulatory_features = get_frmri_articulatory_features(art_f, start, end, stats=stats)
    if articulatory_features is None: continue
    articulatory_features = torch.from_numpy(articulatory_features).unsqueeze(0).unsqueeze(0).to(device)
    if articulatory_features.shape[-2:]!=(136, 136): continue
    art_out, art_z = art_model(articulatory_features, return_latent=True)

    mfccs = get_mfccs(ac_f, start, end)
    if mfccs is None: continue
    mfccs = apply_scaler(mfccs, ac_scalers[subject])
    mfccs = torch.from_numpy(mfccs).unsqueeze(0).to(device)
    ac_out, ac_z = ac_model(mfccs, return_latent=True)

    art_vectors.append(art_z.detach().cpu().numpy().squeeze())
    ac_vectors.append(ac_z.detach().cpu().numpy().squeeze())
    phonemes.append(phone)
    subjects.append(subject)
    groups.append(art_f)

100%|██████████| 30598/30598 [1:09:21<00:00,  7.35it/s]


In [7]:
import numpy as np
art = np.stack(art_vectors, axis=0)   # (N, T_A, D_A)
ac = np.stack(ac_vectors, axis=0)     # (N, T_X, D_X)

# A: (N, T_A, D_A)
# X: (N, T_X, D_X)
N, D = art.shape
N2, D_X = ac.shape
assert N == N2

art_flat = art.reshape(N, D)
ac_flat = ac.reshape(N, D_X)   # shape (N, x_dim)

print("art_flat:", art_flat.shape, "ac_flat:", ac_flat.shape)

art_flat: (30485, 64) ac_flat: (30485, 64)


In [8]:
art_raw = art_flat.astype(np.float32)
ac_raw = ac_flat.astype(np.float32)

In [9]:
from utils import mi

res = mi.compute_all(art_raw, ac_raw, phonemes, groups)
df  = mi.per_phoneme(res)

N=30485  art=64d  ac=64d  phones=37  device=mps

=== art|ac,phone ===
  [art|ac,phone] fold 0: val NLL    -7.87   sigmas at floor  0.0%
  [art|ac,phone] fold 1: val NLL    -3.39   sigmas at floor  0.0%
  [art|ac,phone] fold 2: val NLL   -19.50   sigmas at floor  0.0%
  [art|ac,phone] fold 3: val NLL    -3.62   sigmas at floor  0.0%
  [art|ac,phone] fold 4: val NLL   -14.25   sigmas at floor  0.0%

=== art|phone ===
  [art|phone] fold 0: val NLL     2.94   sigmas at floor  0.0%
  [art|phone] fold 1: val NLL    10.10   sigmas at floor  0.0%
  [art|phone] fold 2: val NLL     8.27   sigmas at floor  0.0%
  [art|phone] fold 3: val NLL    11.67   sigmas at floor  0.0%
  [art|phone] fold 4: val NLL    -2.35   sigmas at floor  0.0%

=== art|ac ===
  [art|ac] fold 0: val NLL    -5.54   sigmas at floor  0.0%
  [art|ac] fold 1: val NLL    -5.47   sigmas at floor  0.0%
  [art|ac] fold 2: val NLL   -13.07   sigmas at floor  0.0%
  [art|ac] fold 3: val NLL    -2.50   sigmas at floor  0.0%
  [art|ac]

In [11]:
df.to_csv("res/frmri.csv", index=False)